# FX-AGENT — Milestone 2 installer (₹0, Google Colab)

**No paid AI API, broker connection or Linux required.** Run cells **in order**. This notebook never runs real money orders. The project demonstration uses synthetic historical-looking prices only, *not* licensed market data or evidence of profitability.

**Before starting:** Download the `FX_AGENT_Milestone2_Addon.zip` from ChatGPT. Open this notebook in Colab via **File → Upload notebook** (the notebook is supplied as a separate `.ipynb` file). Keep your existing Milestone 1 files in your public GitHub repo.

**Safety:** The final publication is **optional**, creates a review branch, and does not touch `main` until you manually merge. Keep `GITHUB_TOKEN` only in Colab Secrets. Never paste it into notebook cells. Colab runtime files disappear when disconnected; GitHub commits persist.

In [ ]:
# CELL 1 — Upload just ONE addon ZIP from Chromebook Downloads.
from google.colab import files
from pathlib import Path
import zipfile, json, hashlib, tempfile, subprocess, sys

incoming = files.upload()
if len(incoming) != 1 or not next(iter(incoming)).endswith('.zip'):
    raise ValueError('Upload exactly FX_AGENT_Milestone2_Addon.zip')
addon_name = next(iter(incoming))
with zipfile.ZipFile(addon_name) as archive:
    file_names = {info.filename for info in archive.infolist() if not info.is_dir()}
    if 'MILESTONE2_MANIFEST.json' not in file_names:
        raise ValueError('Release manifest missing')
    manifest = json.loads(archive.read('MILESTONE2_MANIFEST.json'))
    expected = manifest['files']
    if file_names != set(expected) | {'MILESTONE2_MANIFEST.json'}:
        raise ValueError('Unexpected/missing files in ZIP')
    for info in archive.infolist():
        path = Path(info.filename)
        if path.is_absolute() or '..' in path.parts or chr(92) in info.filename:
            raise ValueError('Unsafe path in addon ZIP')
        if (info.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError('Symlinks are forbidden')
    for name, expected_sha in expected.items():
        if hashlib.sha256(archive.read(name)).hexdigest() != expected_sha:
            raise ValueError('Hash mismatch in ' + name)
print('PASS: Addon ZIP inventory and hashes verified:', len(expected), 'new files')

In [ ]:
# CELL 2 — Clone your PUBLIC Milestone 1 repository, read-only.
REPO = 'unknownopensource/FX-AGENT'
BRANCH = 'main'
project = Path(tempfile.mkdtemp(prefix='fx-agent-m2-')) / 'FX-AGENT'
subprocess.run(
    ['git','clone','--depth','1','--branch',BRANCH,
     f'https://github.com/{REPO}.git',str(project)],
    check=True, timeout=120)
required = ('fxagent/backtest.py', 'fxagent/data.py',
            'fxagent/reporting.py', 'tests/test_foundation.py')
if not all((project / p).is_file() for p in required):
    raise RuntimeError('Your GitHub main branch is missing Milestone 1 files. '
                       'Upload them before proceeding.')
base_commit = subprocess.check_output(
    ['git','rev-parse','HEAD'], cwd=project, text=True).strip()
original_hashes = {
    f: hashlib.sha256((project / f).read_bytes()).hexdigest()
    for f in required}
print('Cloned commit:', base_commit)
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],
               cwd=project, check=True, timeout=120)
print('PASS: Existing Milestone 1 tests')

In [ ]:
# CELL 3 — Add ONLY new M2 files; run the full suite + both demonstrations.
# If any path already exists, stop rather than overwrite existing work.
with zipfile.ZipFile(addon_name) as archive:
    for name in expected:
        target = project / name
        if target.exists():
            raise FileExistsError('Will not overwrite existing file: ' + name)
    for name in expected:
        target = project / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(name))
assert all(hashlib.sha256((project / f).read_bytes()).hexdigest() == h
           for f,h in original_hashes.items()), 'Milestone 1 code changed!'

subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],
               cwd=project, check=True, timeout=120)
subprocess.run([sys.executable,'-m','fxagent.demo'],
               cwd=project, check=True, timeout=120)
subprocess.run([sys.executable,'-m','fxagent.demo_m2'],
               cwd=project, check=True, timeout=120)
tests_passed = True
print('PASS: All tests and both synthetic demonstrations; M1 code unchanged')

## Optional publish: create a review branch in your GitHub repo

Only run the next cell **after all tests pass**. In Colab, open the 🔑 Secrets sidebar; save your GitHub fine-grained token under `GITHUB_TOKEN`, turn on Notebook access. Your token needs **Contents: Read and write** access for `unknownopensource/FX-AGENT` only. No Kie.ai token needed.

The cell checks that GitHub `main` still matches the commit tested above, publishes only the new M2 paths, and opens a compare URL so you can review and create a pull request manually. No existing file is overwritten and nothing is merged automatically.

In [ ]:
# CELL 4 — OPTIONAL: Publish M2 additions to a new branch (no main rewrite).
from google.colab import userdata
from uuid import uuid4
from urllib.parse import quote
import requests

if not globals().get('tests_passed', False):
    raise RuntimeError('Tests not run or failed; publication blocked.')
confirm = input('Type PUBLISH to create a GitHub review branch: ').strip()
if confirm != 'PUBLISH':
    print('Cancelled. No GitHub changes made.')
else:
    token = userdata.get('GITHUB_TOKEN')
    if not token:
        raise RuntimeError('Add GITHUB_TOKEN to Colab Secrets; grant notebook access.')
    session = requests.Session()
    session.headers.update({
        'Authorization': 'Bearer ' + token,
        'Accept': 'application/vnd.github+json',
        'X-GitHub-Api-Version': '2022-11-28',
    })
    api = f'https://api.github.com/repos/{REPO}'
    ref_reply = session.get(f'{api}/git/ref/heads/{BRANCH}', timeout=30)
    ref_reply.raise_for_status()
    current_commit = ref_reply.json()['object']['sha']
    if current_commit != base_commit:
        raise RuntimeError('GitHub main advanced since clone; restart from Cell 2.')

    for name in expected:
        url = f'{api}/contents/' + '/'.join(quote(p, safe='') for p in name.split('/'))
        response = session.get(url, params={'ref': BRANCH}, timeout=30)
        if response.status_code == 200:
            raise RuntimeError('An M2 file already exists on main: ' + name)
        if response.status_code != 404:
            response.raise_for_status()

    new_branch = 'milestone-2-review-' + uuid4().hex[:8]
    create_ref = session.post(f'{api}/git/refs', json={
        'ref': 'refs/heads/' + new_branch, 'sha': base_commit
    }, timeout=30)
    create_ref.raise_for_status()
    print('Created review branch:', new_branch)

    import base64
    with zipfile.ZipFile(addon_name) as archive:
        for name in expected:
            url = f'{api}/contents/' + '/'.join(quote(p, safe='') for p in name.split('/'))
            r = session.put(url, json={
                'message': 'M2: add ' + name,
                'content': base64.b64encode(archive.read(name)).decode('ascii'),
                'branch': new_branch
            }, timeout=60)
            r.raise_for_status()
            print('Added:', name)
    print('All files uploaded to separate branch. Review changes at:')
    print(f'https://github.com/{REPO}/compare/{BRANCH}...{new_branch}?expand=1')
    print('Create and merge a pull request in GitHub only after reviewing.')

## After you merge the pull request

From a new Colab notebook, run these commands to execute the updated public repository without any token or paid APIs:

```python
import subprocess, sys, tempfile
from pathlib import Path
project = Path(tempfile.mkdtemp()) / 'FX-AGENT'
subprocess.run(['git','clone','--depth','1',
                'https://github.com/unknownopensource/FX-AGENT.git', str(project)], check=True)
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],
               cwd=project, check=True)
subprocess.run([sys.executable,'-m','fxagent.demo_m2'],cwd=project,check=True)
```

Do **not** commit Colab cell outputs, API keys, downloaded datasets, synthetic data or generated reports. `main` stays unchanged until you merge a GitHub PR.